#Self-Critique approach
[Paper link](https://proceedings.neurips.cc/paper_files/paper/2023/file/91edff07232fb1b55a505a9e9f6c0ff3-Paper-Conference.pdf)


In [ ]:
import os
import pandas as pd
from openai import OpenAI

# ==========================================
# 1. API AND MODEL CONFIGURATION
# ==========================================

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="YOUR_GROQ_API_KEY_HERE", # Replace with your actual key
)

TUTOR_MODEL = "llama-3.3-70b-versatile"
STUDENT_MODEL = "llama-3.1-8b-instant"

# ==========================================
# 2. THE PROMPTS AND FUNCTIONS
# ==========================================

def generate_initial_hint(problem, correct_solution, student_wrong_answer):
    """Step 1: The Tutor generates the first draft of the hint."""
    system_prompt = """You are an expert, encouraging middle school math tutor.
    Review the Problem, the Correct Solution (your ground truth), and the Student's Incorrect Logic.
    Generate a short, Socratic hint (1-3 sentences) that points the student toward their specific cognitive error.
    Do NOT give away the final answer."""

    user_input = f"Problem: {problem}\nCorrect Solution: {correct_solution}\nStudent's Logic: {student_wrong_answer}"

    response = client.chat.completions.create(
        model=TUTOR_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    # Return both the text and the token count
    return response.choices[0].message.content, response.usage.total_tokens

def critique_hint(problem, correct_solution, student_wrong_answer, current_hint):
    """Step 2: The Tutor critiques its own draft."""
    system_prompt = """You are evaluating a draft of a math homework hint.
    Review the draft against two strict rules:
    1. Does it accidentally give away the final answer or exact calculation?
    2. Is it generic, or does it correctly target the student's specific cognitive error?

    INSTRUCTIONS:
    If the draft violates EITHER of these rules, write a 1-sentence critique of what must be fixed.
    If the draft is completely safe and pedagogically sound, you must output exactly the word: PERFECT. Do not write anything else."""

    user_input = f"Problem: {problem}\nCorrect Solution: {correct_solution}\nStudent's Logic: {student_wrong_answer}\n\nDraft Hint to Critique: {current_hint}"

    response = client.chat.completions.create(
        model=TUTOR_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content.strip(), response.usage.total_tokens

def refine_hint(current_hint, critique):
    """Step 3: The Tutor rewrites the hint based on its own critique."""
    system_prompt = """You are a tutor revising a homework hint.
    Read the original draft and the critique.
    Rewrite the draft hint so that it fixes the issue mentioned in the critique. Keep it to 1-3 sentences and Socratic."""

    user_input = f"Original Draft: {current_hint}\nCritique to Address: {critique}"

    response = client.chat.completions.create(
        model=TUTOR_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.5
    )
    return response.choices[0].message.content, response.usage.total_tokens

def student_final_exam(problem, student_wrong_answer, final_hint):
    """Step 4: The Student Agent takes the final exam to see if the hint actually works."""
    system_prompt = """You are a middle school math student. You tried to solve a problem and got it wrong.
    Your teacher just gave you a final hint.
    Apply the hint to fix your logic.
    CRITICAL INSTRUCTION: Your final output must end with the exact numerical answer. Do not write a long paragraph."""

    user_input = f"Problem: {problem}\nMy Original Wrong Answer: {student_wrong_answer}\nTeacher's Hint: {final_hint}"

    response = client.chat.completions.create(
        model=STUDENT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip(), response.usage.total_tokens

# ==========================================
# 3. THE PIPELINE ORCHESTRATOR
# ==========================================

def run_self_refine_pipeline(problem, correct_solution, student_wrong_answer, max_loops=2):
    print("Starting Approach 1: Self-Refine Pipeline...\n" + "-"*40)

    # Initialize metric trackers for this specific problem
    problem_api_calls = 0
    problem_total_tokens = 0

    # Extract the final correct number
    correct_final_number = str(correct_solution).split("#### ")[-1].strip()

    # --- Step 1: Generate Initial Draft ---
    current_hint, tokens = generate_initial_hint(problem, correct_solution, student_wrong_answer)
    problem_api_calls += 1
    problem_total_tokens += tokens
    print(f"📝 Draft 1: {current_hint}")

    # --- Step 2: The Self-Critique Loop ---
    loops_taken = 0
    for attempt in range(max_loops):
        print(f"\n🔍 Self-Critique Loop {attempt + 1}/{max_loops}...")
        critique, crit_tokens = critique_hint(problem, correct_solution, student_wrong_answer, current_hint)

        problem_api_calls += 1
        problem_total_tokens += crit_tokens

        if "PERFECT" in critique.upper():
            print("✅ Tutor declared the hint PERFECT! Breaking loop early.")
            break
        else:
            print(f"⚠️ Flaw Found: {critique}")
            current_hint, ref_tokens = refine_hint(current_hint, critique)

            problem_api_calls += 1
            problem_total_tokens += ref_tokens
            loops_taken += 1
            print(f"🔄 Revised Hint: {current_hint}")

    # --- Step 3: The Final Exam (Student Agent) ---
    print("\n🧑‍🎓 Handing Final Hint to Student Agent for Exam...")
    student_answer, student_tokens = student_final_exam(problem, student_wrong_answer, current_hint)

    problem_api_calls += 1
    problem_total_tokens += student_tokens

    print(f"Student Output: {student_answer}")

    passed = "Yes" if correct_final_number in student_answer else "No"
    print(f"Student Passed? {passed}")
    print(f"System Cost: {problem_api_calls} API Calls | {problem_total_tokens} Tokens\n" + "="*40)

    # Return the data payload
    return current_hint, loops_taken, passed, problem_api_calls, problem_total_tokens

# ==========================================
# 4. MAIN EXECUTION & EXCEL EXPORT
# ==========================================

if __name__ == "__main__":
    print("Loading dataset...")
    df = pd.read_excel("Phase1_Caregiver_Dataset.xlsx")

    # Create lists to hold our new data
    final_hints = []
    loops_counts = []
    student_results = []
    api_call_totals = []
    token_totals = []

    # Loop through the dataset
    for index, row in df.iterrows():
        print(f"\nProcessing Problem {index + 1} of {len(df)}")

        hint, loops, passed, api_calls, tokens = run_self_refine_pipeline(
            problem=row['Original Problem'],
            correct_solution=row['Correct Answer'],
            student_wrong_answer=row['Student Wrong Answer'],
            max_loops=2 # THIS IS YOUR HYPERPARAMETER
        )

        final_hints.append(hint)
        loops_counts.append(loops)
        student_results.append(passed)
        api_call_totals.append(api_calls)
        token_totals.append(tokens)

    # Save to Excel
    df['Self_Refine_Final_Hint'] = final_hints
    df['Self_Refine_Loops_Taken'] = loops_counts
    df['Student_Passed_Exam'] = student_results
    df['Total_API_Calls'] = api_call_totals
    df['Total_Tokens_Used'] = token_totals

    output_filename = "Approach1_SelfRefine_Results.xlsx"
    df.to_excel(output_filename, index=False)
    print(f"\n🎉 Success! All data saved to {output_filename}")

Loading dataset...

Processing Problem 1 of 50
Starting Approach 1: Self-Refine Pipeline...
----------------------------------------
📝 Draft 1: It looks like you added the eggs Janet eats and bakes to the total number of eggs laid, but shouldn't you be subtracting those since they're not being sold at the market? Think about what happens to the total number of eggs when Janet uses some of them for breakfast and baking.

🔍 Self-Critique Loop 1/2...
⚠️ Flaw Found: The draft hint is too generic and does not specifically target the student's cognitive error of adding instead of subtracting the eaten and baked eggs, and also does not mention the incorrect multiplication by 2 that the student performed.
🔄 Revised Hint: Consider what operation you should perform on the total number of eggs laid when Janet eats or bakes some, and how this differs from the operation you currently have. Additionally, reflect on why multiplying the number of eggs eaten and baked by 2 might not accurately represen

#Multi-agentic simulation Tutor-Student approach
[Paper link](https://proceedings.mlr.press/v264/tonga25a.html)

In [5]:
import os
import time  # Added for rate limit protection
from openai import OpenAI
import pandas as pd

# ==========================================
# 1. API AND MODEL CONFIGURATION
# ==========================================

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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", # Make sure to put your key back!
)

TUTOR_MODEL = "llama-3.3-70b-versatile"
STUDENT_MODEL = "llama-3.1-8b-instant"

# ==========================================
# 2. THE PIPELINE FUNCTIONS
# ==========================================

def generate_tutor_hint(problem, correct_solution, student_wrong_answer):
    system_prompt = """You are an expert, encouraging middle school math tutor.
    Your goal is to help a student fix their mistake without giving away the final answer.
    Review the Problem, the Correct Solution (your ground truth), and the Student's Incorrect Logic.
    Generate a short, Socratic hint (1-3 sentences) that points the student toward their specific cognitive error."""

    user_input = f"""
    Problem: {problem}
    Correct Solution: {correct_solution}
    Student's Incorrect Logic: {student_wrong_answer}
    """

    response = client.chat.completions.create(
        model=TUTOR_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content, response.usage.total_tokens

def simulate_student_validation(problem, student_wrong_answer, tutor_hint):
    system_prompt = """You are a middle school math student. You tried to solve a problem and got it wrong.
    Your teacher just gave you a hint.
    Read the original problem, your original wrong answer, and the teacher's hint.
    Apply the hint to fix your logic.
    CRITICAL INSTRUCTION: Your final output must end with the exact numerical answer. Do not write a long paragraph."""

    user_input = f"""
    Problem: {problem}
    My Original Wrong Answer: {student_wrong_answer}
    Teacher's Hint: {tutor_hint}
    """

    response = client.chat.completions.create(
        model=STUDENT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip(), response.usage.total_tokens

def run_hint_pipeline(problem, correct_solution, student_wrong_answer, max_retries=3):
    print("Starting Cloud API Hint Pipeline...\n" + "="*40)

    problem_api_calls = 0
    problem_total_tokens = 0

    try:
        correct_final_number = correct_solution.split("#### ")[-1].strip()
    except IndexError:
        print("Error parsing correct solution format.")
        return None, "Error", problem_api_calls, problem_total_tokens

    for attempt in range(1, max_retries + 1):
        print(f"\n--- ATTEMPT {attempt} ---")

        # Step A: Generate the Hint
        print("🤖 Tutor is thinking...")
        time.sleep(2) # Rate limit pause
        hint, h_tokens = generate_tutor_hint(problem, correct_solution, student_wrong_answer)
        problem_api_calls += 1
        problem_total_tokens += h_tokens
        print(f"💡 Generated Hint: {hint}\n")

        # Step B: Validate the Hint
        print("🧑‍🎓 Simulated Student is testing the hint...")
        time.sleep(2) # Rate limit pause
        simulated_student_answer, s_tokens = simulate_student_validation(problem, student_wrong_answer, hint)
        problem_api_calls += 1
        problem_total_tokens += s_tokens
        print(f"📝 Simulated Student's New Answer: {simulated_student_answer}\n")

        # Step C: Check for Success
        if correct_final_number in simulated_student_answer:
            print("✅ SUCCESS: The hint helped the student model solve the problem!")
            # RETURNS: Hint, Pass="Yes", API Calls, Tokens
            return hint, "Yes", problem_api_calls, problem_total_tokens
        else:
            print(f"❌ FAILED: The student didn't get '{correct_final_number}'. Regenerating hint...")

    print("\n⚠️ Max retries reached. Returning the safest generated hint from the final attempt.")
    # RETURNS: Hint, Pass="No", API Calls, Tokens
    return hint, "No", problem_api_calls, problem_total_tokens

# ==========================================
# 3. RUNNING A TEST (Main Execution)
# ==========================================

if __name__ == "__main__":
    print("Loading dataset...")
    df = pd.read_excel("Phase1_Caregiver_Dataset.xlsx")

    # Create lists to store all the data
    final_hints_list = []
    student_results = []  # NEW: Tracks Pass/Fail
    api_call_totals = []
    token_totals = []

    for index, row in df.iterrows():
        print(f"\n" + "="*40)
        print(f"Processing Problem {index + 1} of {len(df)}")
        print("="*40)

        # Unpack all FOUR variables
        final_hint, passed, api_calls, tokens = run_hint_pipeline(
            problem=row['Original Problem'],
            correct_solution=str(row['Correct Answer']),
            student_wrong_answer=row['Student Wrong Answer'],
            max_retries=2
        )

        final_hints_list.append(final_hint)
        student_results.append(passed)      # Append Pass/Fail
        api_call_totals.append(api_calls)
        token_totals.append(tokens)

        time.sleep(3) # Rest between rows to protect rate limit

    # Save to Excel
    df['Validation_Final_Hint'] = final_hints_list
    df['Student_Passed_Exam'] = student_results # Pass/Fail Column
    df['Total_API_Calls'] = api_call_totals
    df['Total_Tokens_Used'] = token_totals

    output_filename = "Approach2_ValidationLoop_Results.xlsx"
    df.to_excel(output_filename, index=False)

    print(f"\n🎉 All done! Benchmark results saved to '{output_filename}'")

Loading dataset...

Processing Problem 1 of 50
Starting Cloud API Hint Pipeline...

--- ATTEMPT 1 ---
🤖 Tutor is thinking...
💡 Generated Hint: It seems like you added the eggs Janet eats and bakes to the total number of eggs laid, but shouldn't you be subtracting those since they're not being sold at the market? Think about how many eggs are actually left after Janet uses some for breakfast and baking.

🧑‍🎓 Simulated Student is testing the hint...
📝 Simulated Student's New Answer: Let's fix the logic:

Original problem: Janet's ducks lay 16 eggs per day. She eats 3 for breakfast and bakes 4 muffins.

To find the number of eggs left, we need to subtract the eggs eaten and baked from the total number of eggs laid:
16 (total eggs) - 3 (eaten) - 4 (baked) = 9 eggs left

Janet sells the remaining eggs at $2 per egg. To find the total amount she makes, we multiply the number of eggs left by the price per egg:
9 eggs * $2/egg = $18

So, Janet makes $18 every day at the farmers' market.

✅ SUC

#Multi-agentic simulation Tutor-Student approach (with Knowledge Base)
[Paper link](https://arxiv.org/abs/2502.12633)

In [6]:
import pandas as pd
import json
import time
import re
from openai import OpenAI
from sentence_transformers import SentenceTransformer, util

# --- SECURITY WARNING: Replace with a NEW key, your old one was leaked! ---
GROQ_API_KEY = "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"
client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

print("Loading embedding model (this takes a few seconds the first time)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 1. RAG Knowledge Base
knowledge_base = [
    {
        "error_type": "Order of Operations",
        "description": "Adding before multiplying, ignoring parentheses, or processing left-to-right incorrectly.",
        "strategy": "Ask the student to recite PEMDAS and identify which operation comes first in the specific equation before doing any math."
    },
    {
        "error_type": "Percentage and Ratio Mistakes",
        "description": "Applying a percentage to the wrong base number, or adding percentages directly.",
        "strategy": "Prompt the student to identify 'what is the 100% whole in this scenario?' before applying the percentage."
    },
    {
        "error_type": "Unit Conversion Failure",
        "description": "Mixing up units like inches/feet, hours/minutes, or forgetting to convert before dividing.",
        "strategy": "Ask the student to look at the labels on their numbers. Ask: 'Are these apples and oranges? How do we make the units match?'"
    },
    {
        "error_type": "Reading Comprehension / Missing Steps",
        "description": "Stopping halfway through a problem, answering the wrong question, or forgetting the final step.",
        "strategy": "Ask the student to re-read the very last sentence of the prompt out loud, and ask 'Does our current number answer this specific question?'"
    },
    {
        "error_type": "Basic Arithmetic Setup",
        "description": "Adding instead of multiplying, subtracting instead of adding, or setting up the equation backwards.",
        "strategy": "Ask the student to draw a quick picture or use small substitute numbers (like 2 and 3) to see if their chosen operation makes logical sense."
    }
]

kb_descriptions = [item["description"] for item in knowledge_base]
kb_embeddings = embedder.encode(kb_descriptions)

# 2. RAG Retrieval Function
def retrieve_strategy(caregiver_input):
    query_embedding = embedder.encode(caregiver_input)
    hits = util.semantic_search(query_embedding, kb_embeddings, top_k=1)
    best_match_idx = hits[0][0]['corpus_id']
    return knowledge_base[best_match_idx]['strategy']

# 3. Tutor LLM Function (Generates the Hint)
def generate_rag_hint(model_name, problem, correct_ans, caregiver_input, retrieved_strategy):
    system_prompt = f"""
    You are an expert, empathetic educational copilot for parents.
    CRITICAL TUTORING STRATEGY TO USE: {retrieved_strategy}

    Instructions:
    1. Do NOT reveal the final correct answer or the exact formula.
    2. Read the caregiver's input to understand the child's error.
    3. Apply the CRITICAL TUTORING STRATEGY provided above to formulate your hint.
    4. Use language appropriate for a parent speaking to a 7th grader.

    Output Format: You must output ONLY a valid JSON object with these two keys:
    {{
    "internal_reasoning": "Explain how you applied the required strategy.",
    "caregiver_hint": "The exact script the parent should read to the child."
    }}
    """
    user_prompt = f"Problem: {problem}\nCorrect Answer: {correct_ans}\nCaregiver says: {caregiver_input}"

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.3
        )
        result = json.loads(response.choices[0].message.content)
        # return both the hint AND the token count
        return result.get("caregiver_hint", "Error"), response.usage.total_tokens
    except Exception as e:
        print(f"Tutor Error: {e}")
        return "API Error", 0

# 4. Student LLM Function (Simulates the Child guessing again)
def simulate_student_response(model_name, problem, hint):
    system_prompt = """
    You are a 7th-grade student taking a math test. You previously got the wrong answer.
    Your parent just gave you a helpful hint.
    Use their hint to try and solve the problem again.

    Output Format: You must output ONLY a valid JSON object:
    {
        "thought_process": "How the hint changes your math steps",
        "final_number": "Only the final numeric answer you calculated (e.g., '15', '4.5')"
    }
    """
    user_prompt = f"Problem: {problem}\nParent's Hint: {hint}"

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.5
        )
        result = json.loads(response.choices[0].message.content)
        # return both the answer AND the token count
        return str(result.get("final_number", "")).strip(), response.usage.total_tokens
    except Exception as e:
        print(f"Student Error: {e}")
        return "Error", 0

# Helper function to clean numbers for accurate comparison
def check_answer_match(student_ans, correct_ans):
    s_clean = re.sub(r'[^\d.]', '', str(student_ans))
    c_clean = re.sub(r'[^\d.]', '', str(correct_ans))
    try:
        return float(s_clean) == float(c_clean)
    except:
        return s_clean == c_clean

# 5. Execute the Simulation Loop
print("Loading dataset...")
df = pd.read_excel("Phase1_Caregiver_Dataset.xlsx")

TUTOR_MODEL = "llama-3.3-70b-versatile"
STUDENT_MODEL = "llama-3.1-8b-instant"

# Setup the exact columns you asked for
df["Simulation_Status"] = ""       # Pass/Fail column
df["Attempts_Taken"] = 0
df["Last_Hint_Used"] = ""
df["Last_Strategy_Used"] = ""
df["Total_API_Calls"] = 0          # NEW: API Tracker
df["Total_Tokens_Used"] = 0        # NEW: Token Tracker

print("Starting Multi-Agent Simulation Benchmark...")

for index, row in df.iterrows():
    print(f"\nProcessing Row {index + 1}/50...")

    problem = row["Original Problem"]
    correct_ans = row["Correct Answer"]
    current_caregiver_input = row["Caregiver Input"]

    # Initialize tracking variables for this specific row
    problem_api_calls = 0
    problem_total_tokens = 0

    max_attempts = 2
    status = "Failed" # Defaults to failed
    last_hint = ""
    last_strategy = ""

    # The 2-Attempt Loop
    for attempt in range(1, max_attempts + 1):
        print(f"  Attempt {attempt}...")

        # Step A: Tutor figures out strategy and gives hint
        strategy = retrieve_strategy(current_caregiver_input) # Embedder runs locally (0 tokens)

        time.sleep(2) # Rate limit pause
        hint, t_tokens = generate_rag_hint(TUTOR_MODEL, problem, correct_ans, current_caregiver_input, strategy)

        problem_api_calls += 1
        problem_total_tokens += t_tokens

        last_hint = hint
        last_strategy = strategy

        # Step B: Student tries to use the hint
        time.sleep(2) # Rate limit pause
        student_new_answer, s_tokens = simulate_student_response(STUDENT_MODEL, problem, hint)

        problem_api_calls += 1
        problem_total_tokens += s_tokens

        # Step C: Evaluate Student's Answer
        if check_answer_match(student_new_answer, correct_ans):
            status = "Success" # Marks Pass/Fail correctly
            print(f"    -> Student got it right! ({student_new_answer})")
            break # Exit the loop early since they succeeded!
        else:
            print(f"    -> Student failed again. Guessed: {student_new_answer}")
            current_caregiver_input = f"I gave them your hint, but they tried again and got {student_new_answer}. They are still confused. Can you give a totally different hint?"

    time.sleep(3) # Rest between rows to protect rate limit

    # Save the final results for this row
    df.at[index, "Simulation_Status"] = status
    df.at[index, "Attempts_Taken"] = attempt
    df.at[index, "Last_Hint_Used"] = last_hint
    df.at[index, "Last_Strategy_Used"] = last_strategy
    df.at[index, "Total_API_Calls"] = problem_api_calls       # SAVES API CALLS
    df.at[index, "Total_Tokens_Used"] = problem_total_tokens  # SAVES TOKENS

# 6. Save the final output
output_file = "Approach3_ValidationLoop(KB)_Results.xlsx"
df.to_excel(output_file, index=False)
print(f"\nSuccess! Agentic Simulation Results saved to {output_file}.")

Loading embedding model (this takes a few seconds the first time)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading dataset...
Starting Multi-Agent Simulation Benchmark...

Processing Row 1/50...
  Attempt 1...
    -> Student failed again. Guessed: 10
  Attempt 2...
    -> Student failed again. Guessed: 32

Processing Row 2/50...
  Attempt 1...
    -> Student got it right! (3)

Processing Row 3/50...
  Attempt 1...
    -> Student failed again. Guessed: 48600
  Attempt 2...
    -> Student failed again. Guessed: 65000

Processing Row 4/50...
  Attempt 1...
    -> Student got it right! (540)

Processing Row 5/50...
  Attempt 1...
    -> Student got it right! (20)

Processing Row 6/50...
  Attempt 1...
    -> Student got it right! (64)

Processing Row 7/50...
  Attempt 1...
    -> Student failed again. Guessed: 80
  Attempt 2...
    -> Student got it right! (260)

Processing Row 8/50...
  Attempt 1...
    -> Student failed again. Guessed: 100
  Attempt 2...
    -> Student failed again. Guessed: 240

Processing Row 9/50...
  Attempt 1...
    -> Student failed again. Guessed: 40
  Attempt 2...
   